In [ ]:
# @title run a thing colab
# this is for refrence. You can not use this if you don't have a colab subscription due to ToS.
REPO_URL = "https://github.com/MerchantMan0/IR.git"  # @param {type:"string"}
REPO_DIR = "/content/IR"  # @param {type:"string"}
PROJECT_DIR = "scan-pipeline"  # @param {type:"string"}
ENTRY_PY = "app.py"  # @param {type:"string"}
REQUIREMENTS = "requirements.txt"  # @param {type:"string"}
EXTRA_PIP = "pillow-heif, python-dotenv, rembg"  # @param {type:"string"}
VENV_DIR = "/content/venv"

from pathlib import Path

extra_pip = [p.strip() for p in EXTRA_PIP.split(",") if p.strip()]
project_dir = Path(REPO_DIR) / PROJECT_DIR
requirements_path = project_dir / REQUIREMENTS

print(f"Repo:         {REPO_URL}")
print(f"Project dir:  {project_dir}")
print(f"Entry file:   {project_dir / ENTRY_PY}")
print(f"Requirements: {requirements_path}")

In [ ]:
!pip3 install virtualenv
!virtualenv {VENV_DIR}

In [ ]:
# @title Clone the repo if the project dir is missing. Re-run safe.
!test -d {project_dir} || (rm -rf {REPO_DIR} && git clone --depth 1 {REPO_URL} {REPO_DIR})
!ls -la {project_dir}

In [ ]:
# @title Install project deps into the venv (supports requirements file or directory)
if requirements_path.is_dir():
    req_args = " ".join(f"-r {f}" for f in sorted(requirements_path.glob("*.txt")))
else:
    req_args = f"-r {requirements_path}"
extra = " ".join(extra_pip)
!source {VENV_DIR}/bin/activate; pip3 install {req_args} {extra}

In [ ]:
!source {VENV_DIR}/bin/activate; pip3 list

In [ ]:
from google.colab import userdata

ENV_KEYS = ""  # @param {type:"string"}
SECRET_NAMES = ""  # @param {type:"string"}

bot_dir = project_dir
bot_dir.mkdir(parents=True, exist_ok=True)

keys = [k.strip() for k in ENV_KEYS.split(',')]
secrets = [s.strip() for s in SECRET_NAMES.split(',')]

env_lines = []
for key, secret in zip(keys, secrets):
    if key and secret:
        env_lines.append(f"{key}={userdata.get(secret)}")

(bot_dir / ".env").write_text("\n".join(env_lines) + "\n")
print("Wrote", bot_dir / ".env")

In [ ]:
# @title Run
!source {VENV_DIR}/bin/activate; cd {project_dir} && MPLBACKEND=Agg dotenv -f .env run -- python3 {ENTRY_PY}